# Principal Component Analysis - PCA

In this exercise we will introduce **Principal Component Analysis - PCA**, a technique for reducing the number of dimensions of a dataset. Please checkout out source from [here](https://scikit-learn.org/stable/auto_examples/decomposition/plot_pca_iris.html).

By end of the this exercise, users will practice:
- Applying PCA to a 4-dimensional dataset into 3 and 2 dimensions.
- Measuring explained variance and interpreting what each component means.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pandas.plotting import scatter_matrix

import mpl_toolkits.mplot3d

from sklearn.datasets import load_iris
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

plt.rcParams["figure.figsize"] = (6, 4)

## Dataset

The dataset is part of scikit-learn, so there is nothing to download. PCA is an **unsupervised** method, it will never see the species labels. We only use them afterwards to colour the plots and check whether PCA found structure that matches the real species.

In [ ]:
iris = load_iris(as_frame=True)

features = iris.data                                   
species = iris.target_names[iris.target]              
species_colors = np.array(["tab:blue", "tab:orange", "tab:green"])[iris.target]

print("Total number of rows:", len(features))
features.head(10)

In [ ]:
#view dataset statistics

features.describe().round(2)

In [ ]:
#view pair plot

axes = scatter_matrix(features, c=species_colors, figsize=(9, 9), diagonal="hist", s=25, alpha=0.8)
plt.suptitle("Blue = setosa, orange = versicolor, green = virginica")
plt.show()

## What is PCA?

PCA creates **new features**, called **principal components**, that are linear combinations of the original ones. The components are built so that:

1. The **first component (PC1)** points in the direction in which the data varies the most.
2. The **second component (PC2)** points in the direction of the next largest variation, at a right angle to PC1, and so on.

Because the first components capture most of the variation, we can keep only a few of them and still describe the data well. That is **dimensionality reduction**.

## Reduce 4 dimensions to 3

We apply PCA to the four features, keeping `n_components=3`, and plot every flower in the new 3D space.

- Change `elev` and `azim` in the code to look at the plot from different angles.

In [ ]:
pca_3 = PCA(n_components=3)
X_reduced = pca_3.fit_transform(features)

fig = plt.figure(figsize=(8, 6))
ax = fig.add_subplot(111, projection="3d", elev=-150, azim=110)
scatter = ax.scatter(X_reduced[:, 0], X_reduced[:, 1], X_reduced[:, 2], c=iris.target, s=40)

ax.set(title="First three principal components",
       xlabel="1st Principal Component", ylabel="2nd Principal Component", zlabel="3rd Principal Component")
ax.xaxis.set_ticklabels([]); ax.yaxis.set_ticklabels([]); ax.zaxis.set_ticklabels([])

legend = ax.legend(scatter.legend_elements()[0], iris.target_names.tolist(), loc="upper right", title="Classes")
ax.add_artist(legend)
plt.show()

Each component keeps a fraction of the total variation. We can add the fractions together to get the cumulative explained variance, and plot both in a scree plot.

In [ ]:
pca_all = PCA().fit(features)                    
ratio = pca_all.explained_variance_ratio_
cumulative = np.cumsum(ratio)

print(pd.DataFrame({"explained variance": ratio.round(4), "cumulative": cumulative.round(4)},
                   index=["PC1", "PC2", "PC3", "PC4"]))

plt.bar(range(1, 5), ratio, label="each component")
plt.plot(range(1, 5), cumulative, "o-", c="red", label="cumulative")
plt.axhline(0.95, ls="--", c="grey")
plt.xticks(range(1, 5), ["PC1", "PC2", "PC3", "PC4"])
plt.ylabel("Explained variance ratio"); plt.legend(); plt.title("Scree plot")
plt.show()

Since two components keep almost everything, a simple 2D plot is enough.

In [ ]:
X_2d = PCA(n_components=2).fit_transform(features)

for i, name in enumerate(iris.target_names):
    mask = iris.target == i
    plt.scatter(X_2d[mask, 0], X_2d[mask, 1], label=name, s=30, alpha=0.8)
plt.xlabel("1st Principal Component"); plt.ylabel("2nd Principal Component")
plt.legend(title="Classes"); plt.title("Iris data in 2D after PCA")
plt.show()

PCA looks for the directions of largest variation. If one feature has much larger numbers than the others, it will dominate the first component simply because of its units, not because it is more important. The usual remedy is to standardise the features first, so that every feature has the same scale.

The four Iris features are all in cm and of similar size, so the effect is mild here. Let's check how much it changes the result.

In [ ]:
features_scaled = StandardScaler().fit_transform(features)

ratio_raw = PCA().fit(features).explained_variance_ratio_
ratio_scaled = PCA().fit(features_scaled).explained_variance_ratio_

comparison = pd.DataFrame({"raw features": ratio_raw, "scaled features": ratio_scaled},
                          index=["PC1", "PC2", "PC3", "PC4"]).round(4)
print(comparison)

comparison.plot.bar(rot=0)
plt.ylabel("Explained variance ratio"); plt.title("Effect of scaling")
plt.show()

Each principal component is a represent a weighted sum of the original features. The weights are called loadings, and they are stored in `pca.components_`. A large positive or negative weight means the feature contributes a lot to that component.

In [ ]:
pca_scaled = PCA(n_components=2).fit(features_scaled)

loadings = pd.DataFrame(pca_scaled.components_.T, index=features.columns, columns=["PC1", "PC2"]).round(3)
print(loadings)

loadings.plot.barh()
plt.axvline(0, c="k", lw=0.8); plt.xlabel("Loading"); plt.title("Loadings of the first two components")
plt.show()